In [1]:
import numpy as np
from scipy.spatial import ConvexHull
import json, math


In [10]:
def perimeter_convex(pts):
    if len(pts) < 3:
        return np.nan
    try:
        hull = ConvexHull(pts)
    except Exception:
        return np.nan
    return float(sum(np.linalg.norm(pts[hull.vertices[i]] - pts[hull.vertices[(i+1)%len(hull.vertices)]])
                     for i in range(len(hull.vertices))))

def _perimeter_convex(pts):
    if pts.shape[0] < 3:
        return np.nan
    try:
        hull = ConvexHull(pts)
    except Exception:
        return np.nan
    P = 0.0
    H = hull.vertices
    for i in range(len(H)):
        a = pts[H[i]]
        b = pts[H[(i+1) % len(H)]]
        P += np.linalg.norm(a - b)
    return float(P)


def _slice_points_xy(vertices, y0, tol=0.008):
    """مقطع افقی (fallback)؛ پروجکت به (x,z)."""
    y = vertices[:, 1]
    m = np.abs(y - y0) < tol
    pts = vertices[m][:, [0, 2]]
    if pts.shape[0] < 3:
        return np.empty((0, 2))
    return pts - pts.mean(axis=0)

def slice_points(vertices, y0, tol=0.0075):
    # y = vertices[:, 1]
    # mask = np.abs(y - y0) < tol
    # pts = vertices[mask][:, [0, 2]]
    # return pts - pts.mean(axis=0) if pts.shape[0] >= 3 else np.empty((0,2))
    y = vertices[:, 1]
    m = np.abs(y - y0) < tol
    pts = vertices[m][:, [0, 2]]
    if pts.shape[0] < 3:
        return np.empty((0, 2))
    return pts - pts.mean(axis=0)

def scan_perimeter_between(vertices, y_min, y_max, step=0.005):
    if y_max <= y_min:
        return np.empty((0, 2))
    ys = np.arange(y_min, y_max, step)
    if ys.size == 0:
        ys = np.linspace(y_min, y_max, 3)
    vals = []
    for y0 in ys:
        per = _perimeter_convex(slice_points(vertices, y0))
        vals.append([y0, per])
    return np.asarray(vals, float)

# def orth_slice_along_segment(vertices, p0, p1, t=0.5, radius=0.10, tol=0.010):
#     v = p1 - p0
#     nv = np.linalg.norm(v)
#     if nv < 1e-6: return np.nan
#     v /= nv
#     c = p0 + t * (p1 - p0)
#     X = vertices - c
#     proj = X @ v
#     ortho = X - np.outer(proj, v)
#     dist = np.linalg.norm(ortho, axis=1)
#     tube = dist < radius
#     if not np.any(tube): return np.nan
#     slab = np.abs(proj[tube]) < tol
#     pts = vertices[tube][slab]
#     if pts.shape[0] < 3: return np.nan
#     a = np.array([1.0,0.0,0.0]) if abs(v[0])<0.9 else np.array([0.0,1.0,0.0])
#     u = np.cross(v,a); u /= np.linalg.norm(u)
#     w = np.cross(v,u)
#     pts2 = np.stack([(pts-c)@u, (pts-c)@w], axis=1)
#     return perimeter_convex(pts2)

def orth_slice_along_segment(vertices, p0, p1, t=0.5, radius=0.10, tol=0.008):
    v = p1 - p0
    nv = np.linalg.norm(v)
    if nv < 1e-6:
        return np.nan
    v = v / nv
    c = p0 + t * (p1 - p0)

    X = vertices - c
    proj = X @ v
    ortho = X - np.outer(proj, v)
    dist = np.linalg.norm(ortho, axis=1)

    tube = dist < radius
    if not np.any(tube):
        return np.nan

    slab = np.abs(proj[tube]) < tol
    pts3 = (vertices[tube])[slab]
    if pts3.shape[0] < 3:
        return np.nan

    # صفحه عمود را بساز
    a = np.array([1.0,0.0,0.0])
    if abs(a @ v) > 0.9: a = np.array([0.0,1.0,0.0])
    u = np.cross(v, a); u /= np.linalg.norm(u)
    w = np.cross(v, u)

    rel = pts3 - c
    pts2 = np.stack([rel @ u, rel @ w], axis=1)
    return perimeter_convex(pts2)

def orth_slice_along_segment_robust(vertices, p0, p1,
                                    t_list=(0.30, 0.40, 0.50, 0.60),
                                    radius=0.14, tol=0.012,
                                    horiz_fallback=True):
    """
    مقطع عمود بر محور اندام؛ چند t را امتحان می‌کند و بهترین محیط را برمی‌دارد.
    اگر هیچ کدام جواب نداد و horiz_fallback=True باشد، برش افقی در y میانه را امتحان می‌کند.
    """
    v = p1 - p0
    nv = np.linalg.norm(v)
    if nv < 1e-6:
        return np.nan
    v = v / nv

    # مبنای صفحه عمود (u, w)
    a = np.array([1.0, 0.0, 0.0])
    if abs(a @ v) > 0.9:
        a = np.array([0.0, 1.0, 0.0])
    u = np.cross(v, a); u = u / np.linalg.norm(u)
    w = np.cross(v, u)

    best = np.nan
    for t in t_list:
        c = p0 + t * (p1 - p0)
        X = vertices - c
        proj = X @ v
        ortho = X - np.outer(proj, v)
        dist = np.linalg.norm(ortho, axis=1)

        tube = dist < radius
        if not np.any(tube):
            continue
        slab = np.abs(proj[tube]) < tol
        pts3 = (vertices[tube])[slab]
        if pts3.shape[0] < 3:
            continue

        rel = pts3 - c
        pts2 = np.stack([rel @ u, rel @ w], axis=1)
        per = _perimeter_convex(pts2)
        if np.isfinite(per):
            best = per if not np.isfinite(best) else max(best, per)

    # اگر هیچ برشی جواب نداد، fallback افقی
    if not np.isfinite(best) and horiz_fallback:
        y_mid = 0.5 * (p0[1] + p1[1])
        pts2 = _slice_points_xy(vertices, y_mid, tol=max(0.010, tol))
        per = _perimeter_convex(pts2)
        if np.isfinite(per):
            best = per

    return best


def best_orth_perimeter(vertices, p0, p1, t_list=(0.3,0.4,0.5,0.6), radius=0.12, tol=0.012):
    vals = []
    for t in t_list:
        per = orth_slice_along_segment(vertices, p0, p1, t=t, radius=radius, tol=tol)
        if np.isfinite(per):
            vals.append(per)
    return (max(vals) if vals else np.nan)

In [11]:
#  = {
#     'pelvis':0, 'left_hip':1, 'right_hip':2, 'left_knee':4, 'right_knee':5,
#     'left_ankle':7, 'right_ankle':8, 'neck':12,
#     'left_shoulder':15, 'right_shoulder':16,
#     'left_elbow':17, 'right_elbow':18,
#     'left_wrist':19, 'right_wrist':20,
# }
#
# MP_ALT = {
#     'pelvis':0, 'left_hip':1, 'right_hip':2, 'left_knee':3, 'right_knee':4,
#     'left_ankle':5, 'right_ankle':6, 'neck':12,
#     'left_shoulder':13, 'right_shoulder':14,
#     'left_elbow':15, 'right_elbow':16,
#     'left_wrist':17, 'right_wrist':18,
# }
#
# MP_TRUE = {
#     'pelvis': 0,
#     'left_hip': 1, 'right_hip': 2,
#     'left_knee': 4, 'right_knee': 5,
#     'left_ankle': 7, 'right_ankle': 8,
#     'neck': 12,
#     'left_shoulder': 16, 'right_shoulder': 17,
#     'left_elbow': 18, 'right_elbow': 19,
#     'left_wrist': 20, 'right_wrist': 21,
# }

def sleeve_len_cm_from_map(joints, mp):
    L = np.linalg.norm(joints[mp["left_shoulder"]]-joints[mp["left_elbow"]]) + \
        np.linalg.norm(joints[mp["left_elbow"]]-joints[mp["left_wrist"]])
    R = np.linalg.norm(joints[mp["right_shoulder"]]-joints[mp["right_elbow"]]) + \
        np.linalg.norm(joints[mp["right_elbow"]]-joints[mp["right_wrist"]])
    return float(np.nanmean([L, R])*100.0)

def sleeve_len_robust_cm(joints, mp_default, mp_alt=None):
    # بازو از دو نگاشت: default (CUSTOM) و یک ALT متداول
    def seg_len_cm(mp):
        L = np.linalg.norm(joints[mp["left_shoulder"]]-joints[mp["left_elbow"]]) \
          + np.linalg.norm(joints[mp["left_elbow"]]-joints[mp["left_wrist"]])
        R = np.linalg.norm(joints[mp["right_shoulder"]]-joints[mp["right_elbow"]]) \
          + np.linalg.norm(joints[mp["right_elbow"]]-joints[mp["right_wrist"]])
        return float(np.nanmean([L, R]) * 100.0)

    s_default = seg_len_cm(mp_default)

    if mp_alt is None:
        mp_alt = {
            # ALT بازو/مچ/آرنجِ سبک SMPL-24 رایج
            "left_shoulder": 16, "right_shoulder": 17,
            "left_elbow": 18,    "right_elbow": 19,
            "left_wrist": 20,    "right_wrist": 21,
        } | mp_default  # بقیه کلیدها را از mp_default بگیر

    s_alt = seg_len_cm(mp_alt)

    # اگر یکی‌شان در بازهٔ معقول [40, 90] بود همان را بگیر؛ وگرنه نزدیک‌ترین به این بازه
    def score(s):
        if 40.0 <= s <= 90.0: return 0.0
        # فاصله تا بازه
        if s < 40.0: return 40.0 - s
        return s - 90.0

    return s_default if score(s_default) <= score(s_alt) else s_alt



def outseam_cm_from_map(joints, vertices, mp):
    waist_y = 0.5*(joints[mp["left_hip"],1]+joints[mp["right_hip"],1]) + 0.02
    ground_y = min(float(joints[mp["left_ankle"],1]), float(joints[mp["right_ankle"],1]), float(vertices[:,1].min()))
    return float(max(0.0,(waist_y-ground_y))*100.0)

def choose_joint_map_auto(joints, vertices):
    # candidates=[("SMPL24",MP_SMPL24),("ALT",MP_ALT)]
    # scored=[]
    # for name,mp in candidates:
    #     s=sleeve_len_cm_from_map(joints,mp)
    #     o=outseam_cm_from_map(joints,vertices,mp)
    #     bad=int(not(45<=s<=75))+int(not(90<=o<=115))
    #     score=-(bad*1000+abs(s-60)+abs(o-100))
    #     scored.append((score,name,mp,s,o))
    # scored.sort(key=lambda x:-x[0])
    # best=scored[0]
    # print(f"Picked map: {best[1]} | sleeve≈{best[3]:.1f} | outseam≈{best[4]:.1f}")
    # return best[2]
     # مستقیماً نگاشت درست را برگردان
    # print(">>> Using fixed SMPL24 joint mapping (manual override).")
    # return MP_TRUE
    # print(">>> Using CUSTOM joint map for your SMPL variant.")
    # return {
    #     'pelvis': 0,
    #     'left_hip': 1,
    #     'right_hip': 2,
    #     'left_knee': 3,
    #     'right_knee': 4,
    #     'left_ankle': 7,
    #     'right_ankle': 5,
    #     'neck': 22,
    #     'left_shoulder': 13,
    #     'right_shoulder': 14,
    #     'left_elbow': 10,
    #     'right_elbow': 11,
    #     'left_wrist': 19,
    #     'right_wrist': 20,
    # }

    print(">>> Using CUSTOM joint map (pelvis=9, hips=8/12, knees=3/4, ankles=7/6, neck=22).")
    return {
        'pelvis': 9,
        'left_hip': 8,   'right_hip': 12,
        'left_knee': 3,  'right_knee': 4,
        'left_ankle': 7, 'right_ankle': 6,
        'neck': 22,
        'left_shoulder': 13, 'right_shoulder': 14,
        'left_elbow': 10,    'right_elbow': 11,
        'left_wrist': 19,    'right_wrist': 20,
    }

def choose_joint_map_fixed_custom():
    # CUSTOM map (بر اساس چاپ yها)
    return {
        "pelvis": 9,
        "left_hip": 8,  "right_hip": 12,
        "left_knee": 3, "right_knee": 4,
        "left_ankle": 7,"right_ankle": 6,
        "neck": 22,
        "left_shoulder": 20, "right_shoulder": 21,
        "left_elbow": 18,    "right_elbow": 19,
        "left_wrist": 16,    "right_wrist": 17,
    }


# def ground_y_from_ankles(joints, mp):
#     return float(min(joints[mp["left_ankle"],1], joints[mp["right_ankle"],1]))

def ground_y_from_ankles_or_mesh(joints, vertices, mp):
    ankles = float(min(joints[mp["left_ankle"], 1], joints[mp["right_ankle"], 1]))
    floor  = float(vertices[:, 1].min())
    return min(ankles, floor)


In [12]:
def body_height_y(vertices):
    return float(vertices[:,1].max()-vertices[:,1].min())

def rescale_to_user_height(vertices, joints, user_height_m, width_scale=1.0):
    h = body_height_y(vertices)
    s = user_height_m / max(h, 1e-6)
    V = vertices * s
    J = joints   * s
    if abs(width_scale - 1.0) > 1e-6:
        V = V.copy()
        V[:, [0, 2]] *= float(width_scale)
    return V, J, s

def avg_y_mp(joints,mp,a,b):
    return 0.5*(joints[mp[a],1]+joints[mp[b],1])


In [13]:
# def measure_core(vertices,joints):
#     res={}
#     mp=choose_joint_map_auto(joints,vertices)
#
#     y_neck=float(joints[mp["neck"],1])
#     y_pelvis=float(joints[mp["pelvis"],1])
#     y_mid=0.5*(y_neck+y_pelvis)
#
#     scan_chest=scan_perimeter_between(vertices,y_mid,y_neck-0.02,step=0.004)
#     scan_waist=scan_perimeter_between(vertices,y_pelvis+0.03,y_mid,step=0.004)
#     scan_hip=scan_perimeter_between(vertices,y_pelvis-0.10,y_pelvis+0.04,step=0.004)
#     res["chest_circum_cm"]=float(np.nanmax(scan_chest[:,1])*100) if scan_chest.size else np.nan
#     res["waist_circum_cm"]=float(np.nanmin(scan_waist[:,1])*100) if scan_waist.size else np.nan
#     res["hip_circum_cm"]=float(np.nanmax(scan_hip[:,1])*100) if scan_hip.size else np.nan
#
#     # بازو
#     L_sh,L_el=joints[mp["left_shoulder"]],joints[mp["left_elbow"]]
#     R_sh,R_el=joints[mp["right_shoulder"]],joints[mp["right_elbow"]]
#     def best_circ(p0,p1):
#         vals=[orth_slice_along_segment(vertices,p0,p1,t=t,radius=0.12,tol=0.010) for t in (0.35,0.5,0.65)]
#         vals=[v for v in vals if np.isfinite(v)]
#         return max(vals) if vals else np.nan
#     cL,cR=best_circ(L_sh,L_el),best_circ(R_sh,R_el)
#     res["upperarm_circum_cm"]=float(np.nanmean([cL,cR])*100)
#
#     # ران
#     L_hp,L_kn=joints[mp["left_hip"]],joints[mp["left_knee"]]
#     R_hp,R_kn=joints[mp["right_hip"]],joints[mp["right_knee"]]
#     def best_thigh(p0,p1):
#         vals=[orth_slice_along_segment(vertices,p0,p1,t=t,radius=0.20,tol=0.012) for t in (0.4,0.5,0.6)]
#         vals=[v for v in vals if np.isfinite(v)]
#         return max(vals) if vals else np.nan
#     cTL,cTR=best_thigh(L_hp,L_kn),best_thigh(R_hp,R_kn)
#     res["thigh_circum_cm"]=float(np.nanmean([cTL,cTR])*100)
#
#     # قد شلوار و آستین
#     waist_y=avg_y_mp(joints,mp,"left_hip","right_hip")+0.02
#     ground_y=min(float(joints[mp["left_ankle"],1]),float(joints[mp["right_ankle"],1]),float(vertices[:,1].min()))
#     res["pants_outseam_cm"]=float(max(0.0,(waist_y-ground_y))*100)
#     L_len=np.linalg.norm(L_sh-L_el)+np.linalg.norm(L_el-joints[mp["left_wrist"]])
#     R_len=np.linalg.norm(R_sh-R_el)+np.linalg.norm(R_el-joints[mp["right_wrist"]])
#     res["sleeve_len_cm"]=float(np.nanmean([L_len,R_len])*100)
#
#     res["top_len_cm"]=float((y_neck-(y_pelvis+0.02))*100)
#     res["gown_len_cm"]=float(max(0.0,(y_neck-ground_y))*100)
#     return res

# def measure_core(vertices, joints):
#     res = {}
#     mp = choose_joint_map_auto(joints, vertices)
#
#     # تنه
#     y_neck   = float(joints[mp["neck"], 1])
#     # لگن را از ایندکسی بگیر که واقعاً نزدیک کمر است (mp['pelvis']=9)
#     y_pelvis = float(joints[mp["pelvis"], 1])
#     y_mid    = 0.5*(y_neck + y_pelvis)
#
#     # مقاطع افقی برای سینه/کمر/باسن
#     scan_chest = scan_perimeter_between(vertices, y_mid, y_neck - 0.02, step=0.004)
#     if scan_chest.size:
#         i = int(np.nanargmax(scan_chest[:,1])); win = scan_chest[max(0,i-2):i+3,1]
#         res["chest_circum_cm"] = float(np.nanmean(win)*100)
#     else:
#         res["chest_circum_cm"] = np.nan
#
#     scan_waist = scan_perimeter_between(vertices, y_pelvis + 0.03, y_mid, step=0.004)
#     if scan_waist.size:
#         i = int(np.nanargmin(scan_waist[:,1])); win = scan_waist[max(0,i-2):i+3,1]
#         res["waist_circum_cm"] = float(np.nanmean(win)*100)
#     else:
#         res["waist_circum_cm"] = np.nan
#
#     scan_hip = scan_perimeter_between(vertices, y_pelvis - 0.08, y_pelvis + 0.05, step=0.004)
#     if scan_hip.size:
#         i = int(np.nanargmax(scan_hip[:,1])); win = scan_hip[max(0,i-2):i+3,1]
#         res["hip_circum_cm"] = float(np.nanmean(win)*100)
#     else:
#         res["hip_circum_cm"] = np.nan
#
#     # بازو (شانه→آرنج)، با جست‌وجو روی t و radius کمی بزرگ‌تر
#     L_sh, L_el = joints[mp["left_shoulder"]],  joints[mp["left_elbow"]]
#     R_sh, R_el = joints[mp["right_shoulder"]], joints[mp["right_elbow"]]
#     c_bi_L = best_orth_perimeter(vertices, L_sh, L_el, t_list=(0.4,0.5,0.6), radius=0.09, tol=0.007)
#     c_bi_R = best_orth_perimeter(vertices, R_sh, R_el, t_list=(0.4,0.5,0.6), radius=0.09, tol=0.007)
#     res["upperarm_circum_cm"] = float(np.nanmean([c_bi_L, c_bi_R])*100)
#
#     # ران (هیپ→زانو)
#     L_hp, L_kn = joints[mp["left_hip"]],  joints[mp["left_knee"]]
#     R_hp, R_kn = joints[mp["right_hip"]], joints[mp["right_knee"]]
#     c_th_L = best_orth_perimeter(vertices, L_hp, L_kn, t_list=(0.45,0.55,0.65), radius=0.12, tol=0.008)
#     c_th_R = best_orth_perimeter(vertices, R_hp, R_kn, t_list=(0.45,0.55,0.65), radius=0.12, tol=0.008)
#     res["thigh_circum_cm"] = float(np.nanmean([c_th_L, c_th_R])*100)
#
#     # قد شلوار: از میانگین هیپ‌ها + ۲cm تا پایین‌ترین مچ پا (عمودی)
#     y_side_waist = 0.5*(joints[mp["left_hip"],1] + joints[mp["right_hip"],1]) + 0.02
#     y_ankles_min = ground_y_from_ankles(joints, mp)
#     res["pants_outseam_cm"] = float(max(0.0, y_side_waist - y_ankles_min)*100)
#
#     # قد آستین: شانه→آرنج→مچ (طول مسیر)
#     L_len = np.linalg.norm(L_sh - L_el) + np.linalg.norm(L_el - joints[mp["left_wrist"]])
#     R_len = np.linalg.norm(R_sh - R_el) + np.linalg.norm(R_el - joints[mp["right_wrist"]])
#     res["sleeve_len_cm"] = float(np.nanmean([L_len, R_len])*100)
#
#     # قد بلوز: گردن تا بالای لگن (pelvis + 2cm)
#     y_neck_base = y_neck
#     y_hip_top   = y_pelvis + 0.02
#     res["top_len_cm"]  = float(max(0.0, y_neck_base - y_hip_top)*100)
#
#     # قد پیراهن: گردن تا پایین‌ترین مچ پا
#     res["gown_len_cm"] = float(max(0.0, y_neck_base - y_ankles_min)*100)
#
#     return res

# Cell 3: measurement core

# def measure_core(vertices, joints):
#     res = {}
#     mp = choose_joint_map_fixed_custom()
#     print(">>> Using CUSTOM joint map (pelvis=9, hips=8/12, knees=3/4, ankles=7/6, neck=22).")
#
#     y_neck   = float(joints[mp["neck"],   1])
#     y_pelvis = float(joints[mp["pelvis"], 1])
#     y_mid    = 0.5 * (y_neck + y_pelvis)
#
#     # Chest
#     scan_chest = scan_perimeter_between(vertices, y_mid, y_neck - 0.02, step=0.004)
#     if scan_chest.size:
#         idx = int(np.nanargmax(scan_chest[:, 1]))
#         win = scan_chest[max(0, idx-2): idx+3, 1]
#         res["chest_circum_cm"] = float(np.nanmean(win) * 100.0)
#     else:
#         res["chest_circum_cm"] = np.nan
#
#     # Waist
#     scan_waist = scan_perimeter_between(vertices, y_pelvis + 0.03, y_mid, step=0.004)
#     if scan_waist.size:
#         idx = int(np.nanargmin(scan_waist[:, 1]))
#         win = scan_waist[max(0, idx-2): idx+3, 1]
#         res["waist_circum_cm"] = float(np.nanmean(win) * 100.0)
#     else:
#         res["waist_circum_cm"] = np.nan
#
#     # Hip (کران پایین را کمی پایین‌تر می‌بریم تا باسن واقعی‌تر شود)
#     scan_hip = scan_perimeter_between(vertices, y_pelvis - 0.12, y_pelvis + 0.05, step=0.004)
#     if scan_hip.size:
#         idx = int(np.nanargmax(scan_hip[:, 1]))
#         win = scan_hip[max(0, idx-2): idx+3, 1]
#         res["hip_circum_cm"] = float(np.nanmean(win) * 100.0)
#     else:
#         res["hip_circum_cm"] = np.nan
#
#     # Upper arm (orthogonal slice, robust)
#     L_sh, L_el = joints[mp["left_shoulder"]],  joints[mp["left_elbow"]]
#     R_sh, R_el = joints[mp["right_shoulder"]], joints[mp["right_elbow"]]
#
#     c_bi_L = orth_slice_along_segment_robust(vertices, L_sh, L_el,
#                                              t_list=(0.35, 0.45, 0.55),
#                                              radius=0.14, tol=0.012, horiz_fallback=True)
#     c_bi_R = orth_slice_along_segment_robust(vertices, R_sh, R_el,
#                                              t_list=(0.35, 0.45, 0.55),
#                                              radius=0.14, tol=0.012, horiz_fallback=True)
#     # میانگین بازو
#     res["upperarm_circum_cm"] = float(np.nanmean([c_bi_L, c_bi_R]) * 100.0)
#
#     # Thigh (orth slice، بهترین از چند t)
#     L_hp, L_kn = joints[mp["left_hip"]],  joints[mp["left_knee"]]
#     R_hp, R_kn = joints[mp["right_hip"]], joints[mp["right_knee"]]
#
#     def best_thigh(vertices, p0, p1):
#         vals = [
#             orth_slice_along_segment_robust(vertices, p0, p1, t_list=(0.40,), radius=0.18, tol=0.012),
#             orth_slice_along_segment_robust(vertices, p0, p1, t_list=(0.50,), radius=0.18, tol=0.012),
#             orth_slice_along_segment_robust(vertices, p0, p1, t_list=(0.60,), radius=0.18, tol=0.012),
#         ]
#         vals = [v for v in vals if np.isfinite(v)]
#         return (max(vals) if vals else np.nan)
#
#     c_th_L = best_thigh(vertices, L_hp, L_kn)
#     c_th_R = best_thigh(vertices, R_hp, R_kn)
#     res["thigh_circum_cm"] = float(np.nanmean([c_th_L, c_th_R]) * 100.0)
#
#     # Outseam: میانگین هیپ‌ها + 5cm تا زمین
#     y_side_waist = avg_y_mp(joints, mp, "left_hip", "right_hip") + 0.05
#     y_ground     = ground_y_from_ankles_or_mesh(joints, vertices, mp)
#     res["pants_outseam_cm"] = float(max(0.0, y_side_waist - y_ground) * 100.0)
#
#     # Sleeve length (دقیق)
#     L_len = np.linalg.norm(L_sh - L_el) + np.linalg.norm(L_el - joints[mp["left_wrist"]])
#     R_len = np.linalg.norm(R_sh - R_el) + np.linalg.norm(R_el - joints[mp["right_wrist"]])
#     res["sleeve_len_cm"] = float(np.nanmean([L_len, R_len]) * 100.0)
#
#     # Top / Gown
#     y_floor_mesh = float(vertices[:, 1].min())
#     res["top_len_cm"]  = float((y_neck - (y_pelvis + 0.02)) * 100.0)
#     res["gown_len_cm"] = float(max(0.0, y_neck - y_floor_mesh) * 100.0)
#
#     return res

def measure_core(vertices, joints):
    res = {}
    mp = choose_joint_map_fixed_custom()
    print(">>> Using CUSTOM joint map (pelvis=9, hips=8/12, knees=3/4, ankles=7/6, neck=22).")

    y_neck   = float(joints[mp["neck"],   1])
    y_pelvis = float(joints[mp["pelvis"], 1])
    y_mid    = 0.5 * (y_neck + y_pelvis)

    # --- Chest
    scan_chest = scan_perimeter_between(vertices, y_mid, y_neck - 0.02, step=0.004)
    if scan_chest.size:
        idx = int(np.nanargmax(scan_chest[:, 1]))
        win = scan_chest[max(0, idx-2): idx+3, 1]
        res["chest_circum_cm"] = float(np.nanmean(win) * 100.0)
    else:
        res["chest_circum_cm"] = np.nan

    # --- Waist
    scan_waist = scan_perimeter_between(vertices, y_pelvis + 0.03, y_mid, step=0.004)
    if scan_waist.size:
        idx = int(np.nanargmin(scan_waist[:, 1]))
        win = scan_waist[max(0, idx-2): idx+3, 1]
        res["waist_circum_cm"] = float(np.nanmean(win) * 100.0)
    else:
        res["waist_circum_cm"] = np.nan

    # --- Hip: حولِ میانگین y هیپ‌ها (واقعی‌تر از y_pelvis)
    y_hips_mean = avg_y_mp(joints, mp, "left_hip", "right_hip")
    scan_hip = scan_perimeter_between(vertices, y_hips_mean - 0.18, y_hips_mean + 0.05, step=0.004)
    if scan_hip.size:
        idx = int(np.nanargmax(scan_hip[:, 1]))
        win = scan_hip[max(0, idx-2): idx+3, 1]
        res["hip_circum_cm"] = float(np.nanmean(win) * 100.0)
    else:
        res["hip_circum_cm"] = np.nan

    # --- Upper arm: انتخاب نگاشت بهتر (پایین آوردن ریسک 167cm)
    res["upperarm_circum_cm"] = float(np.nanmean([
        orth_slice_along_segment_robust(vertices, joints[mp["left_shoulder"]],  joints[mp["left_elbow"]],
                                        t_list=(0.35, 0.45, 0.55), radius=0.14, tol=0.012, horiz_fallback=True),
        orth_slice_along_segment_robust(vertices, joints[mp["right_shoulder"]], joints[mp["right_elbow"]],
                                        t_list=(0.35, 0.45, 0.55), radius=0.14, tol=0.012, horiz_fallback=True)
    ]) * 100.0)

    # --- Thigh
    def best_thigh(vertices, p0, p1):
        vals = [
            orth_slice_along_segment_robust(vertices, p0, p1, t_list=(0.40,), radius=0.18, tol=0.012),
            orth_slice_along_segment_robust(vertices, p0, p1, t_list=(0.50,), radius=0.18, tol=0.012),
            orth_slice_along_segment_robust(vertices, p0, p1, t_list=(0.60,), radius=0.18, tol=0.012),
        ]
        vals = [v for v in vals if np.isfinite(v)]
        return (max(vals) if vals else np.nan)

    L_hp, L_kn = joints[mp["left_hip"]],  joints[mp["left_knee"]]
    R_hp, R_kn = joints[mp["right_hip"]], joints[mp["right_knee"]]
    c_th_L = best_thigh(vertices, L_hp, L_kn)
    c_th_R = best_thigh(vertices, R_hp, R_kn)
    res["thigh_circum_cm"] = float(np.nanmean([c_th_L, c_th_R]) * 100.0)

    # --- Outseam: از «کمر جانبی» (میانگین هیپ‌ها + 2cm) تا زمین واقعی
    y_side_waist = y_hips_mean + 0.02
    y_ground     = ground_y_from_ankles_or_mesh(joints, vertices, mp)
    res["pants_outseam_cm"] = float(max(0.0, y_side_waist - y_ground) * 100.0)

    # --- Sleeve: دو نگاشت امتحان و انتخاب منطقی
    mp_alt = {
        "left_shoulder": 16, "right_shoulder": 17,
        "left_elbow": 18,    "right_elbow": 19,
        "left_wrist": 20,    "right_wrist": 21,
        # بقیه همان CUSTOM
        "pelvis": mp["pelvis"], "left_hip": mp["left_hip"], "right_hip": mp["right_hip"],
        "left_knee": mp["left_knee"], "right_knee": mp["right_knee"],
        "left_ankle": mp["left_ankle"], "right_ankle": mp["right_ankle"],
        "neck": mp["neck"],
    }
    res["sleeve_len_cm"] = sleeve_len_robust_cm(joints, mp, mp_alt)

    # --- Top/Gown: با زمین واقعی و کمر جانبی
    res["top_len_cm"]  = float(max(0.0, y_neck - (y_hips_mean + 0.02)) * 100.0)
    res["gown_len_cm"] = float(max(0.0, y_neck - y_ground) * 100.0)

    return res



In [14]:
# def measure_pipeline(smpl_output_tpose,user_height_cm,gender,width_scale=0.75):
#     vertices=smpl_output_tpose["vertices"]
#     joints=smpl_output_tpose["joints"]
#     vertices,joints,s=rescale_to_user_height(vertices,joints,user_height_cm/100.0,width_scale)
#     print("Scaled body height (m):",body_height_y(vertices))
#     res=measure_core(vertices,joints)
#     res.update({"gender":gender,"user_height_cm":user_height_cm,
#                 "scale_factor":s,"width_scale":width_scale})
#     return res

# Cell 4: pipeline (run)

def measure_pipeline(smpl_output_tpose, user_height_cm=175, gender="male", width_scale=0.75):
    vertices = smpl_output_tpose["vertices"]
    joints   = smpl_output_tpose["joints"]

    vertices, joints, s = rescale_to_user_height(vertices, joints, user_height_cm/100.0, width_scale)
    print("Scaled body height (m):", body_height_y(vertices))
    print(">>> Using CUSTOM joint map (pelvis=9, hips=8/12, knees=3/4, ankles=7/6, neck=22).")

    res = measure_core(vertices, joints)
    res.update({
        "gender": gender,
        "user_height_cm": user_height_cm,
        "scale_factor": s,
        "width_scale": width_scale,
    })
    return res



In [15]:
# Cell: نمایش فارسی نتایج اندازه‌گیری

def print_measurement_results_farsi(results):
    """چاپ فارسی و خوانا برای دیکشنری نتایج اندازه‌گیری بدن."""
    labels = {
        "chest_circum_cm": "دور سینه",
        "waist_circum_cm": "دور کمر",
        "hip_circum_cm": "دور باسن",
        "upperarm_circum_cm": "دور بازو",
        "thigh_circum_cm": "دور ران",
        "pants_outseam_cm": "قد بیرونی شلوار",
        "sleeve_len_cm": "قد آستین",
        "top_len_cm": "قد بالاتنه (بلوز)",
        "gown_len_cm": "قد لباس بلند",
        "user_height_cm": "قد کاربر",
        "scale_factor": "ضریب مقیاس مدل",
        "width_scale": "ضریب پهنای بدن",
        "gender": "جنسیت",
    }

    print("📏 نتایج اندازه‌گیری بدن:\n" + "-"*40)
    for key, label in labels.items():
        if key not in results:
            continue
        val = results[key]
        if isinstance(val, (int, float)) and np.isfinite(val):
            # اگر واحد سانتی‌متر است
            if key.endswith("_cm"):
                print(f"{label}: {val:.1f} سانتی‌متر")
            elif key == "scale_factor":
                print(f"{label}: {val:.3f}")
            elif key == "width_scale":
                print(f"{label}: {val:.2f}")
            else:
                print(f"{label}: {val}")
        elif isinstance(val, str):
            print(f"{label}: {val}")
        else:
            print(f"{label}: نامشخص")

    print("-"*40)


In [8]:
tp = np.load("logs/demo/img_triplet_/tmp_tpose_smpl.npz", allow_pickle=True)

results = measure_pipeline(tp, user_height_cm=175, gender="male", width_scale=0.75)
print_measurement_results_farsi(results)

Scaled body height (m): 1.75
>>> Using CUSTOM joint map (pelvis=9, hips=8/12, knees=3/4, ankles=7/6, neck=22).
>>> Using CUSTOM joint map (pelvis=9, hips=8/12, knees=3/4, ankles=7/6, neck=22).
📏 نتایج اندازه‌گیری بدن:
----------------------------------------
دور سینه: 75.6 سانتی‌متر
دور کمر: 57.6 سانتی‌متر
دور باسن: 68.3 سانتی‌متر
دور بازو: 40.3 سانتی‌متر
دور ران: 44.9 سانتی‌متر
قد بیرونی شلوار: 89.8 سانتی‌متر
قد آستین: 167.8 سانتی‌متر
قد بالاتنه (بلوز): 84.1 سانتی‌متر
قد لباس بلند: 173.9 سانتی‌متر
قد کاربر: 175.0 سانتی‌متر
ضریب مقیاس مدل: 0.926
ضریب پهنای بدن: 0.75
جنسیت: male
----------------------------------------


In [9]:
import numpy as np

data = np.load("logs/demo/img_triplet_/tmp_tpose_smpl.npz", allow_pickle=True)
joints = data["joints"]

# چاپ فقط محور y (ارتفاع) برای 0 تا 23
for i, y in enumerate(joints[:24,1]):
    print(f"{i:2d}: {y: .4f}")


 0: -0.8895
 1: -0.7533
 2: -0.6978
 3: -0.7164
 4: -0.9403
 5: -0.7047
 6: -0.7285
 7: -0.9444
 8: -0.2290
 9: -0.1379
10:  0.2529
11:  0.6449
12: -0.1397
13:  0.2407
14:  0.6258
15: -0.9225
16: -0.9234
17: -0.8866
18: -0.8917
19:  0.7278
20:  0.7187
21:  0.6468
22:  0.7448
23:  0.7372
